# Pan-cancer Hist2Pheno 课题总结

结果分两块，都已经在 `data/` 里。数字在 §9–§10 从文件读，不手写。

1. **共享三级细胞注释**（§1–§8）。训练队列是 7 个 dataset：**Lung、BRCA、CRC、HCC、PDAC、GIST、GBM**。训练表是 [`PanCancerCellType.xlsx`](PanCancerCellType.xlsx) 的 `celltype_pancancer`：**L1 五类、L12 八类、L2 二十类**。`celltype_initial` 是并表库存，`celltype_fine` 是更细的对照。
2. **Pan-cancer 模型**（§9–§10）。当前权重是 **UNI2 A2**（`best_mlp_gpu.pt` 等于 fold 1）。内部是 5-fold group CV；训练队列上是这一个权重的 deployment 预测，StarDist 核与 GT 匹配后的 AUROC；没有细胞类型标签的 Visium **NPC**（两张）和 **BRCA**（四张）用同一个权重。

| 块 | 路径 |
|---|---|
| 注释表 | `code/Hist2Pheno_pkg/dataset/PanCancerCellType.xlsx` |
| UNI2 A2 训练 / 内部 CV | `data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/` |
| UNI2 A2 deployment（896 张训练切片） | `data/PanCancer/pred_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42_deployment/` |
| NPC external | `data/Visium/NPC/pred_uni2_A2_bestmlp_paired_v2_seed42/` |
| BRCA external | `data/Visium/BRCA/pred_uni2_A2_bestmlp_paired_v2_seed42/` |

External 和 deployment 用的是同一个文件：`data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt`。`pred_all_spatial/` 和 `pred_pancancer_section/` 是旧 UNI `D_emph_L2`，这一节不读。

---

## 注释怎么并（§1–§8）

下面的交并集、presence、Venn、UpSet、Jaccard 都直接在这 **7 个训练 dataset** 上算。七个的 L2 并集就是训练头里的 20 类。

Lung 的细类收成粗类后进入同一张表：气道上皮 → `Epithelial`，稳态 FB → `Fibroblast`；肺泡、损伤上皮、活化成纤维单独留成 `Alveolar`、`Injury_epithelial`、`Myofibroblast`。CRC 的 Flex / RCTD Label1 也映射进这 20 类，不新开 L2。P1、P2、P5 都有转移后的标签。


In [ ]:
from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)
from matplotlib_venn import venn2, venn3

HERE = Path(".").resolve()
if not (HERE / "PanCancerCellType.xlsx").exists():
    HERE = Path("/home/lingyu/ssd2/Python/Hist2Pheno/code/Hist2Pheno_pkg/dataset")
XLSX = HERE / "PanCancerCellType.xlsx"
FIG = HERE / "figures_l2_overlap"
FIG.mkdir(exist_ok=True)

# Training cohorts, same order as the pan-cancer CV.
TRAIN = ["Lung", "BRCA", "CRC", "HCC", "PDAC", "GIST", "GBM"]
COLORS = {
    "Lung": "#7f8c8d",
    "BRCA": "#c0392b",
    "CRC": "#117a65",
    "HCC": "#d35400",
    "PDAC": "#1e8449",
    "GIST": "#2471a3",
    "GBM": "#6c3483",
}
# Coarse classes added from Lung. Myofibroblast is also used by CRC.
LUNG_COARSE = {"Alveolar", "Injury_epithelial", "Myofibroblast"}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.use14corefonts": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(XLSX, "exists=", XLSX.exists())
print("training datasets:", ", ".join(TRAIN))


## 1. 读表，构造每个 dataset 的 L2 集合

`L2_v1` 是映射进训练头之后的类。`L2` 是各 dataset 自己的 native 名（可训练的那些）。


In [ ]:
def trainable_l2(df: pd.DataFrame, col: str = "L2_v1") -> set[str]:
    mask = df.get("trainable_v1", "Y") == "Y"
    vals = df.loc[mask, col].dropna().astype(str).str.strip()
    return {v for v in vals if v and v not in {"Other", "nan"}}


def native_trainable_l2(df: pd.DataFrame) -> set[str]:
    mask = df.get("trainable", "Y") == "Y"
    vals = df.loc[mask, "L2"].dropna().astype(str).str.strip()
    return {v for v in vals if v and v not in {"Other", "nan"}}


sheets = {name: pd.read_excel(XLSX, name) for name in TRAIN}
sets_l2 = {name: trainable_l2(sheets[name]) for name in TRAIN}
sets_native = {name: native_trainable_l2(sheets[name]) for name in TRAIN}

count_tbl = pd.DataFrame({
    "dataset": TRAIN,
    "native_L2": [len(sets_native[n]) for n in TRAIN],
    "L2": [len(sets_l2[n]) for n in TRAIN],
})
display(count_tbl)

print("L2 集合（训练头）:")
for n in TRAIN:
    print(f"  {n:5s} ({len(sets_l2[n]):2d}): {sorted(sets_l2[n])}")


## 2. 七个训练 dataset 的交并集

- **7 交集**：七个 dataset 都有的类
- **7 并集**：训练头里的 20 类
- **k-way**：恰好出现在 k 个 dataset 的类


In [ ]:
def set_stats(sets: dict[str, set[str]], names: list[str]) -> dict:
    vals = [sets[n] for n in names]
    union = set.union(*vals)
    inter = set.intersection(*vals)
    return {"union": union, "intersect": inter}


seven = set_stats(sets_l2, TRAIN)
seven_native = set_stats(sets_native, TRAIN)

print(f"L2  7-union      ({len(seven['union']):2d}): {sorted(seven['union'])}")
print(f"L2  7-intersect  ({len(seven['intersect']):2d}): {sorted(seven['intersect'])}")
print(f"native 7-union   ({len(seven_native['union']):2d})")
print(f"native 7-intersect ({len(seven_native['intersect']):2d}): {sorted(seven_native['intersect'])}")

labels = sorted(seven["union"])
presence = pd.DataFrame(
    {ds: [int(lab in sets_l2[ds]) for lab in labels] for ds in TRAIN},
    index=labels,
)
presence["n_datasets"] = presence[TRAIN].sum(axis=1)
presence = presence.sort_values(["n_datasets"] + TRAIN, ascending=[False] + [False] * len(TRAIN))
display(presence)

print("恰好出现在 k 个 dataset:")
for k, grp in presence.groupby("n_datasets"):
    print(f"  k={k} ({len(grp)}): {list(grp.index)}")


## 3. Presence 热图

行是 7 个训练 dataset 的 L2 并集。浅色行是从 Lung 收进来的三个粗类。


In [ ]:
def _presence_heatmap(ax, df, datasets, title, extra_rows=None):
    mat = df[datasets].to_numpy()
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(datasets)))
    ax.set_xticklabels(datasets)
    ylabels = []
    for lab in df.index:
        ylabels.append(f"{lab} *" if extra_rows and lab in extra_rows else lab)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(ylabels)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, "●" if mat[i, j] else "", ha="center", va="center", color="white", fontsize=9)
        ax.text(len(datasets) + 0.15, i, str(int(df.iloc[i]["n_datasets"])), va="center", fontsize=9)
        if extra_rows and df.index[i] in extra_rows:
            ax.axhspan(i - 0.5, i + 0.5, color="#fdebd0", alpha=0.35, zorder=0)
    ax.set_xlim(-0.5, len(datasets) + 0.85)
    ax.axvline(len(datasets) - 0.5, color="0.7", lw=0.8)


fig, ax = plt.subplots(figsize=(5.2, 5.2))
_presence_heatmap(
    ax, presence, TRAIN,
    "L2 presence · 7 training datasets",
    extra_rows=LUNG_COARSE,
)
ax.text(
    1.0, -0.06, "* coarse class added from Lung (Myofibroblast is also in CRC)",
    transform=ax.transAxes, ha="right", va="top", fontsize=8, color="#b9770e",
)
fig.tight_layout()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "01_presence_heatmap_7datasets.pdf", bbox_inches="tight")
plt.show()


## 4. Venn

七圆 Venn 读不清，拆成训练队列里的三组：

- Lung ∩ BRCA ∩ CRC（三个 Xenium）
- HCC ∩ PDAC ∩ GIST（三个 CODEX 实体瘤 / TMA）
- 其余六个的并集 vs GBM


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.2, 5.0))
venn3(
    [sets_l2["Lung"], sets_l2["BRCA"], sets_l2["CRC"]],
    set_labels=("Lung", "BRCA", "CRC"),
    set_colors=(COLORS["Lung"], COLORS["BRCA"], COLORS["CRC"]),
    alpha=0.45,
    ax=axes[0],
)
axes[0].set_title("L2 · Lung ∩ BRCA ∩ CRC")
venn3(
    [sets_l2["HCC"], sets_l2["PDAC"], sets_l2["GIST"]],
    set_labels=("HCC", "PDAC", "GIST"),
    set_colors=(COLORS["HCC"], COLORS["PDAC"], COLORS["GIST"]),
    alpha=0.45,
    ax=axes[1],
)
axes[1].set_title("L2 · HCC ∩ PDAC ∩ GIST")
rest = set.union(*(sets_l2[n] for n in TRAIN if n != "GBM"))
venn2(
    [rest, sets_l2["GBM"]],
    set_labels=("Lung∪BRCA∪CRC∪HCC∪PDAC∪GIST", "GBM"),
    set_colors=("#1abc9c", COLORS["GBM"]),
    alpha=0.5,
    ax=axes[2],
)
axes[2].set_title("L2 · other six vs GBM")
fig.tight_layout()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "02_venn_train7.png", bbox_inches="tight")
plt.show()

print("Lung ∩ BRCA ∩ CRC =", sorted(sets_l2["Lung"] & sets_l2["BRCA"] & sets_l2["CRC"]))
print("HCC ∩ PDAC ∩ GIST =", sorted(sets_l2["HCC"] & sets_l2["PDAC"] & sets_l2["GIST"]))
print("GBM only:", sorted(sets_l2["GBM"] - rest))
print("six only vs GBM:", sorted(rest - sets_l2["GBM"]))


## 5. UpSet：七个训练 dataset 的互斥交集

每个柱是「恰好这几个 dataset 共有、且不在其余 dataset」的 L2 个数。


In [ ]:
def exclusive_intersections(sets: dict[str, set[str]], names: list[str]):
    rows = []
    for r in range(1, len(names) + 1):
        for comb in combinations(names, r):
            inter = set.intersection(*(sets[n] for n in comb))
            others = [n for n in names if n not in comb]
            extra = set.union(*(sets[n] for n in others)) if others else set()
            excl = inter - extra
            if excl:
                rows.append({"combo": comb, "labels": sorted(excl), "n": len(excl)})
    rows.sort(key=lambda d: (-d["n"], -len(d["combo"]), d["combo"]))
    return rows


upset_rows = exclusive_intersections(sets_l2, TRAIN)
upset_df = pd.DataFrame(
    {
        "datasets": [" ∩ ".join(r["combo"]) for r in upset_rows],
        "n": [r["n"] for r in upset_rows],
        "L2": [", ".join(r["labels"]) for r in upset_rows],
    }
)
display(upset_df)

n_bar = len(upset_rows)
xs = np.arange(n_bar)
set_sizes = [len(sets_l2[n]) for n in TRAIN]
degree = [len(r["combo"]) for r in upset_rows]
bar_colors = plt.cm.Blues(np.interp(degree, [1, len(TRAIN)], [0.35, 0.85]))

fig = plt.figure(figsize=(8.6, 4.8))
gs = fig.add_gridspec(
    2, 2,
    width_ratios=[1.9, 5.2],
    height_ratios=[2.3, 2.4],
    wspace=0.04,
    hspace=0.08,
)
ax_bar = fig.add_subplot(gs[0, 1])
ax_mat = fig.add_subplot(gs[1, 1], sharex=ax_bar)
ax_set = fig.add_subplot(gs[1, 0])

ax_bar.bar(xs, [r["n"] for r in upset_rows], color=bar_colors, width=0.62, edgecolor="white")
for i, r in enumerate(upset_rows):
    ax_bar.text(i, r["n"] + 0.08, str(r["n"]), ha="center", va="bottom", fontsize=9, fontweight="bold")
ax_bar.set_ylabel("# exclusive L2")
ax_bar.set_title("UpSet · L2 exclusive intersections (7 training datasets)")
ax_bar.set_ylim(0, max(r["n"] for r in upset_rows) + 0.9)
ax_bar.tick_params(axis="x", bottom=False, labelbottom=False)
ax_bar.spines["bottom"].set_visible(False)

for i in range(len(TRAIN)):
    ax_mat.axhline(i, color="#e8e8e8", lw=6, zorder=0, solid_capstyle="butt")
for j, r in enumerate(upset_rows):
    idxs = [i for i, name in enumerate(TRAIN) if name in r["combo"]]
    if len(idxs) >= 2:
        ax_mat.plot([j, j], [min(idxs), max(idxs)], color="#2c3e50", lw=2.2, zorder=1)
    for i, name in enumerate(TRAIN):
        on = name in r["combo"]
        ax_mat.scatter(
            j, i, s=78, zorder=2,
            facecolors="#1a252f" if on else "#d0d3d4",
            edgecolors="#1a252f" if on else "#b0b3b5",
            linewidths=0.6,
        )
ax_mat.set_yticks(range(len(TRAIN)))
ax_mat.set_yticklabels([])
ax_mat.set_xticks(xs)
ax_mat.set_xticklabels(
    [", ".join(r["labels"]) for r in upset_rows],
    rotation=90,
    ha="center",
    va="top",
    fontsize=8,
)
ax_mat.set_xlim(-0.6, n_bar - 0.4)
ax_mat.set_ylim(-0.7, len(TRAIN) - 0.3)
ax_mat.invert_yaxis()
ax_mat.tick_params(length=0)
for sp in ax_mat.spines.values():
    sp.set_visible(False)

ax_set.barh(range(len(TRAIN)), set_sizes, color=[COLORS[n] for n in TRAIN], height=0.55)
ax_set.set_yticks(range(len(TRAIN)))
ax_set.set_yticklabels([f"{n} ({s})" for n, s in zip(TRAIN, set_sizes)], fontsize=9)
ax_set.set_ylim(-0.7, len(TRAIN) - 0.3)
ax_set.invert_yaxis()
ax_set.invert_xaxis()
ax_set.set_xlim(max(set_sizes) + 1.2, 0)
ax_set.set_xticks([])
ax_set.set_xlabel("set size", fontsize=8)
ax_set.tick_params(left=False, right=False, labelright=False, length=0)
for sp in ax_set.spines.values():
    sp.set_visible(False)

fig.subplots_adjust(bottom=0.28)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "04_upset_7datasets.pdf", bbox_inches="tight")
plt.show()


## 6. Pairwise Jaccard

两张热图比的是七个训练 dataset 的 **L2 类名集合**，不是细胞数。每个集合来自对应 sheet 里 `trainable_v1 == Y` 的 `L2_v1`，空值和 `Other` 不计入。行、列都是 Lung、BRCA、CRC、HCC、PDAC、GIST、GBM。

- **左图 Jaccard**：格子 `(A, B)` 是 `|A ∩ B| / |A ∪ B|`。两个 dataset 的 L2 类名完全一样是 1，没有共有类是 0。对角线按定义写成 1.00。例如 Lung 有 16 类、BRCA 有 11 类，共有 8 类，并集有 19 类，所以 Lung–BRCA 是 8/19 = 0.42。
- **右图 |A ∩ B|**：非对角线是两个 dataset 共有的 L2 类数。对角线是该 dataset 自己的 L2 类数：Lung 16、BRCA 11、CRC 15、HCC 10、PDAC 11、GIST 11、GBM 6。


In [ ]:
jacc = pd.DataFrame(np.nan, index=TRAIN, columns=TRAIN)
overlap_n = pd.DataFrame(0, index=TRAIN, columns=TRAIN)
for a, b in combinations(TRAIN, 2):
    inter = sets_l2[a] & sets_l2[b]
    uni = sets_l2[a] | sets_l2[b]
    j = len(inter) / len(uni) if uni else 0
    jacc.loc[a, b] = jacc.loc[b, a] = j
    overlap_n.loc[a, b] = overlap_n.loc[b, a] = len(inter)
for a in TRAIN:
    jacc.loc[a, a] = 1.0
    overlap_n.loc[a, a] = len(sets_l2[a])

n = len(TRAIN)
fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.2))
im0 = axes[0].imshow(jacc.to_numpy(), cmap="YlGnBu", vmin=0, vmax=1)
axes[0].set_xticks(range(n)); axes[0].set_xticklabels(TRAIN, rotation=40, ha="right")
axes[0].set_yticks(range(n)); axes[0].set_yticklabels(TRAIN)
axes[0].set_title("Jaccard (L2) · 7 training datasets")
for i in range(n):
    for j in range(n):
        axes[0].text(j, i, f"{jacc.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(overlap_n.to_numpy(), cmap="Oranges")
axes[1].set_xticks(range(n)); axes[1].set_xticklabels(TRAIN, rotation=40, ha="right")
axes[1].set_yticks(range(n)); axes[1].set_yticklabels(TRAIN)
axes[1].set_title("|A ∩ B| (L2) · 7 training datasets")
for i in range(n):
    for j in range(n):
        axes[1].text(j, i, str(int(overlap_n.iloc[i, j])), ha="center", va="center", fontsize=8)
fig.colorbar(im1, ax=axes[1], fraction=0.046)
fig.tight_layout()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "05_pairwise_jaccard_7datasets.png", bbox_inches="tight")
plt.show()
display(overlap_n)


## 7. 二十类训练头

七个 dataset 的 L2 并集等于 `celltype_initial` 里可训练的 20 类。表里的 `l2_scope` 仍沿用当时的标记：`core_5union` 是早期五个癌种就有的类，`lung_unique` 是 Lung 收成粗类后加进去的三个。CRC 的 Flex / RCTD Label1 映射进这些类，不另开 L2。

| Lung native（已合并，不单列） | → L2 |
|------------------------------|------|
| Basal / Secretory / Goblet / Multiciliated / PNEC / Prolif Airway / Mesothelial | `Epithelial` |
| Alveolar / Adventitial / Subpleural FBs | `Fibroblast` |
| CD4 / CD8 / Treg / NK-NKT / B / Plasma / DC / Mac / Neutrophil / Mast / 内皮亚型 | 对应类（Treg→`Tregs`，NK 与未分型 T→`NK`） |

| 从 Lung 留下的粗类 | 收进去的 native |
|--------------------|-----------------|
| `Alveolar` | AT1, AT2, Proliferating AT2 |
| `Injury_epithelial` | KRT5-/KRT17+, RASC, Transitional AT2 |
| `Myofibroblast` | Myofibroblasts, Activated / Inflammatory / Proliferating FBs（CRC 的 Myofibroblast / Proliferating Fibroblast 也进这一类） |


In [ ]:
ct_sheet = pd.read_excel(XLSX, "celltype_initial")
cols = [c for c in ["L1", "L12", "L2", "l2_scope", "present_in"] if c in ct_sheet.columns]
train = ct_sheet[ct_sheet["trainable"].astype(str).str.upper() == "Y"][cols].copy()
display(train.reset_index(drop=True))
print("trainable L2 =", len(train))
assert set(train["L2"].astype(str)) == seven["union"]
print("OK: 7-union = trainable L2")

crc = sheets["CRC"].copy()
crc = crc[crc.get("trainable_v1", "Y") == "Y"]
crc_map = (
    crc.dropna(subset=["L2_v1"])
    .groupby(["L1_v1", "L12_v1", "L2_v1"], dropna=False)["celltype_level2"]
    .apply(lambda s: "; ".join(sorted({str(x).strip() for x in s if str(x).strip() and str(x) != "Unlabeled"})))
    .reset_index()
    .rename(columns={"L1_v1": "L1", "L12_v1": "L12", "L2_v1": "L2", "celltype_level2": "Flex_native"})
)
display(crc_map.sort_values(["L1", "L12", "L2"]).reset_index(drop=True))
new_l2 = sets_l2["CRC"] - seven["union"]
print("CRC 是否引入训练头之外的 L2:", sorted(new_l2) or "(none)")
assert not new_l2, new_l2
print("figures →", FIG)


## 8. Link three-level cell type annotation using Sankey diagram

Sheet `celltype_initial`：L1 (5) → L12 (8) → L2 (20)。丢掉 `Other`。每条 L2 权重 1，父节点带宽 = 子类个数。
Interactive Plotly Sankey + static PNG（`figures_l2_overlap/09_ontology_sankey.*`）。


In [ ]:
from collections import defaultdict

from matplotlib.patches import FancyBboxPatch, PathPatch
from matplotlib.path import Path as MPath

L1_ORDER_SK = ["Epithelial", "Immune", "Stromal", "Endothelial", "Neural"]
L12_ORDER_SK = [
    "Tumor", "Epithelial", "T_cell", "B_Plasma",
    "Myeloid", "Stromal", "Endothelial", "Neural",
]
L1_COLOR_SK = {
    "Epithelial": "#E67E22",
    "Immune": "#2E86C1",
    "Stromal": "#27AE60",
    "Endothelial": "#B7950B",
    "Neural": "#8E44AD",
}


def _hex_rgb(h: str) -> tuple[float, float, float]:
    h = h.lstrip("#")
    return tuple(int(h[i : i + 2], 16) / 255 for i in (0, 2, 4))


def _hex_rgba(h: str, a: float) -> str:
    r, g, b = (int(h.lstrip("#")[i : i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{a})"


def _label_color(hex_color: str) -> str:
    r, g, b = _hex_rgb(hex_color)
    return "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.62 else "white"


onto = pd.read_excel(XLSX, "celltype_initial")
onto = onto.loc[
    (onto["trainable"].astype(str).str.upper() == "Y")
    & (onto["L2"].astype(str).str.strip().str.lower() != "other")
].copy()
onto["L1"] = pd.Categorical(onto["L1"], L1_ORDER_SK, ordered=True)
onto["L12"] = pd.Categorical(onto["L12"], L12_ORDER_SK, ordered=True)
onto = onto.sort_values(["L1", "L12", "L2"]).reset_index(drop=True)
assert len(onto) == 20, f"expected 20 L2, got {len(onto)}: {onto['L2'].tolist()}"

l1_list = [x for x in L1_ORDER_SK if x in set(onto["L1"].astype(str))]
l12_list = [x for x in L12_ORDER_SK if x in set(onto["L12"].astype(str))]
l2_list = onto["L2"].astype(str).tolist()
l12_to_l1 = (
    onto.drop_duplicates("L12")
    .assign(_k=lambda d: d["L12"].astype(str))
    .set_index("_k")["L1"]
    .astype(str)
    .to_dict()
)
l2_to_l1 = dict(zip(onto["L2"].astype(str), onto["L1"].astype(str)))
l2_to_l12 = dict(zip(onto["L2"].astype(str), onto["L12"].astype(str)))

print(f"Sankey nodes  L1={len(l1_list)}  L12={len(l12_list)}  L2={len(l2_list)}  (Other dropped)")
display(onto[["L1", "L12", "L2", "l2_scope"]].reset_index(drop=True))

# --- Plotly interactive Sankey ---
try:
    import plotly.graph_objects as go

    def _nid(level: str, name: str) -> str:
        return f"{level}:{name}"

    nodes = (
        [_nid("L1", x) for x in l1_list]
        + [_nid("L12", x) for x in l12_list]
        + [_nid("L2", x) for x in l2_list]
    )
    idx = {k: i for i, k in enumerate(nodes)}
    labels = l1_list + l12_list + l2_list
    node_color = (
        [L1_COLOR_SK[x] for x in l1_list]
        + [L1_COLOR_SK[l12_to_l1[x]] for x in l12_list]
        + [L1_COLOR_SK[l2_to_l1[x]] for x in l2_list]
    )
    src, tgt, val, lcol = [], [], [], []
    for l12, g in onto.groupby("L12", observed=True):
        l1 = str(g["L1"].iloc[0])
        src.append(idx[_nid("L1", l1)])
        tgt.append(idx[_nid("L12", str(l12))])
        val.append(len(g))
        lcol.append(_hex_rgba(L1_COLOR_SK[l1], 0.45))
    for _, r in onto.iterrows():
        l1, l12, l2 = str(r["L1"]), str(r["L12"]), str(r["L2"])
        src.append(idx[_nid("L12", l12)])
        tgt.append(idx[_nid("L2", l2)])
        val.append(1)
        lcol.append(_hex_rgba(L1_COLOR_SK[l1], 0.35))

    def _stacked_y(names, group_of, order_groups, gap=0.035):
        n_gap = sum(1 for g in order_groups if any(group_of[n] == g for n in names)) - 1
        unit = (1.0 - gap * max(n_gap, 0)) / len(names)
        y, cursor = {}, 1.0
        for g in order_groups:
            members = [n for n in names if group_of[n] == g]
            if not members:
                continue
            for i, n_ in enumerate(members):
                y[n_] = cursor - unit * (i + 0.5)
            cursor -= unit * len(members) + gap
        return y

    l2_y = _stacked_y(l2_list, l2_to_l1, l1_list)
    l12_y = {
        l12: float(np.mean([l2_y[k] for k in l2_list if l2_to_l12[k] == l12]))
        for l12 in l12_list
    }
    l1_y = {
        l1: float(np.mean([l12_y[k] for k in l12_list if l12_to_l1[k] == l1]))
        for l1 in l1_list
    }
    xs = [0.01] * len(l1_list) + [0.38] * len(l12_list) + [0.86] * len(l2_list)
    ys = [l1_y[x] for x in l1_list] + [l12_y[x] for x in l12_list] + [l2_y[x] for x in l2_list]
    fig_sk = go.Figure(
        go.Sankey(
            arrangement="fixed",
            node=dict(
                pad=10,
                thickness=18,
                line=dict(color="white", width=1),
                label=labels,
                color=node_color,
                x=xs,
                y=ys,
            ),
            link=dict(source=src, target=tgt, value=val, color=lcol),
        )
    )
    fig_sk.update_layout(
        title="Pan-cancer cell type  ·  L1 (5) → L12 (8) → L2 (20)",
        font=dict(size=13),
        height=720,
        width=980,
        margin=dict(l=10, r=10, t=50, b=20),
    )
    fig_sk.write_html(FIG / "09_ontology_sankey.html", include_plotlyjs="cdn")
    fig_sk.show()
except ImportError:
    print("plotly not installed; static alluvial PNG only")

# --- Static alluvial (same flows; saved PNG) ---
def _stacked_boxes(names, weights, y0=0.02, y1=0.98, gap=0.012):
    total = sum(weights)
    usable = (y1 - y0) - gap * max(len(names) - 1, 0)
    out, cursor = {}, y1
    for n, w in zip(names, weights):
        h = usable * (w / total)
        out[n] = (cursor - h, cursor)
        cursor -= h + gap
    return out


def _child_slices(parent_boxes, parent_of, children, child_w):
    groups = defaultdict(list)
    for ch in children:
        groups[parent_of[ch]].append(ch)
    out = {}
    for p, chs in groups.items():
        ya, yb = parent_boxes[p]
        h = yb - ya
        tw = sum(child_w[c] for c in chs)
        gap = min(0.006, h * 0.04 / max(len(chs), 1))
        usable = h - gap * (len(chs) - 1) if len(chs) > 1 else h
        cursor = yb
        for c in chs:
            hh = usable * (child_w[c] / tw)
            out[c] = (cursor - hh, cursor)
            cursor -= hh + gap
    return out


def _ribbon(ax, x0, y0a, y0b, x1, y1a, y1b, color, alpha=0.38):
    xm = (x0 + x1) / 2
    path = MPath(
        [
            (x0, y0a), (xm, y0a), (xm, y1a), (x1, y1a),
            (x1, y1b), (xm, y1b), (xm, y0b), (x0, y0b), (x0, y0a),
        ],
        [
            MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
            MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.CLOSEPOLY,
        ],
    )
    ax.add_patch(PathPatch(path, facecolor=color, edgecolor="none", alpha=alpha, zorder=1))


cols_x = {"L1": (0.02, 0.18), "L12": (0.40, 0.56), "L2": (0.78, 0.96)}
l1_w = onto.groupby("L1", observed=True).size().reindex(l1_list).tolist()
l12_w = onto.groupby("L12", observed=True).size().reindex(l12_list).tolist()
l2_w = [1] * len(l2_list)
b1 = _stacked_boxes(l1_list, l1_w, gap=0.028)
b12 = _stacked_boxes(l12_list, l12_w, gap=0.016)
b2 = _stacked_boxes(l2_list, l2_w, gap=0.008)
l12_wmap = dict(zip(l12_list, l12_w))
l2_wmap = dict(zip(l2_list, l2_w))
l12_in_l1 = _child_slices(b1, l12_to_l1, l12_list, l12_wmap)
l2_in_l12 = _child_slices(b12, l2_to_l12, l2_list, l2_wmap)

fig, ax = plt.subplots(figsize=(11.2, 8.2))
ax.set_xlim(0, 1)
ax.set_ylim(-0.02, 1.08)
ax.axis("off")
ax.set_title("Pan-cancer cell type  ·  L1 (5)  –  L12 (8)  –  L2 (20)", fontsize=13, pad=8)

x1r, (x12l, x12r), x2l = cols_x["L1"][1], cols_x["L12"], cols_x["L2"][0]
for l12 in l12_list:
    ya, yb = l12_in_l1[l12]
    ya2, yb2 = b12[l12]
    _ribbon(ax, x1r, ya, yb, x12l, ya2, yb2, _hex_rgb(L1_COLOR_SK[l12_to_l1[l12]]), 0.42)
for l2 in l2_list:
    ya, yb = l2_in_l12[l2]
    ya2, yb2 = b2[l2]
    _ribbon(ax, x12r, ya, yb, x2l, ya2, yb2, _hex_rgb(L1_COLOR_SK[l2_to_l1[l2]]), 0.32)

for col, names, boxes, color_of, fs in [
    ("L1", l1_list, b1, {k: L1_COLOR_SK[k] for k in l1_list}, 10),
    ("L12", l12_list, b12, {k: L1_COLOR_SK[l12_to_l1[k]] for k in l12_list}, 9),
    ("L2", l2_list, b2, {k: L1_COLOR_SK[l2_to_l1[k]] for k in l2_list}, 8),
]:
    x0, x1 = cols_x[col]
    for n in names:
        ya, yb = boxes[n]
        c = color_of[n]
        ax.add_patch(
            FancyBboxPatch(
                (x0, ya), x1 - x0, yb - ya,
                boxstyle="round,pad=0.004,rounding_size=0.01",
                facecolor=c, edgecolor="white", linewidth=1.2, zorder=3,
            )
        )
        ax.text(
            (x0 + x1) / 2, (ya + yb) / 2, n,
            ha="center", va="center", fontsize=fs,
            color=_label_color(c), fontweight="medium", zorder=4,
        )

for col, lab in zip(["L1", "L12", "L2"], ["L1  (5)", "L12  (8)", "L2  (20)"]):
    x0, x1 = cols_x[col]
    ax.text((x0 + x1) / 2, 1.03, lab, ha="center", va="bottom", fontsize=11, fontweight="bold", color="#333")

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "09_ontology_sankey.png", bbox_inches="tight", facecolor="white")
plt.show()
print("figures →", FIG / "09_ontology_sankey.png")


In [ ]:
import sys

# Same static alluvial as the cell above. Box fills are the current pan-cancer
# palette: L1 coarse, L12 intermediate, L2 fine. Ribbons take the color of the
# class they enter.
_repo = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _p in (_repo / "code", _repo / "code" / "Hist2Pheno_pkg"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
from plotting_palettes import get_palette

L1_PAL = get_palette("pancancer", "coarse")
L12_PAL = get_palette("pancancer", "intermediate")
L2_PAL = get_palette("pancancer", "fine")
box_color = {"L1": L1_PAL, "L12": L12_PAL, "L2": L2_PAL}
for level, names, pal in (
    ("L1", l1_list, L1_PAL),
    ("L12", l12_list, L12_PAL),
    ("L2", l2_list, L2_PAL),
):
    missing = [n for n in names if n not in pal]
    if missing:
        raise KeyError(f"{level} missing from pancancer palette: {missing}")


def _ink(rgba) -> str:
    r, g, b = rgba[:3]
    return "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.62 else "white"


l1_w = onto.groupby("L1", observed=True).size().reindex(l1_list).tolist()
l12_w = onto.groupby("L12", observed=True).size().reindex(l12_list).tolist()
l2_w = [1] * len(l2_list)
b1 = _stacked_boxes(l1_list, l1_w, gap=0.028)
b12 = _stacked_boxes(l12_list, l12_w, gap=0.016)
b2 = _stacked_boxes(l2_list, l2_w, gap=0.008)
l12_in_l1 = _child_slices(b1, l12_to_l1, l12_list, dict(zip(l12_list, l12_w)))
l2_in_l12 = _child_slices(b12, l2_to_l12, l2_list, dict(zip(l2_list, l2_w)))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.set_xlim(0, 1)
ax.set_ylim(-0.02, 1.08)
ax.axis("off")
ax.set_title(
    "Pan-cancer cell type  ·  L1 (5)  –  L12 (8)  –  L2 (20)",
    fontsize=13,
    pad=8,
)

x1r, (x12l, x12r), x2l = cols_x["L1"][1], cols_x["L12"], cols_x["L2"][0]
for l12 in l12_list:
    ya, yb = l12_in_l1[l12]
    ya2, yb2 = b12[l12]
    _ribbon(ax, x1r, ya, yb, x12l, ya2, yb2, L12_PAL[l12][:3], 0.42)
for l2 in l2_list:
    ya, yb = l2_in_l12[l2]
    ya2, yb2 = b2[l2]
    _ribbon(ax, x12r, ya, yb, x2l, ya2, yb2, L2_PAL[l2][:3], 0.32)

for col, names, boxes, fs in [
    ("L1", l1_list, b1, 10),
    ("L12", l12_list, b12, 9),
    ("L2", l2_list, b2, 8),
]:
    x0, x1 = cols_x[col]
    pal = box_color[col]
    for n in names:
        ya, yb = boxes[n]
        c = pal[n]
        ax.add_patch(
            FancyBboxPatch(
                (x0, ya), x1 - x0, yb - ya,
                boxstyle="round,pad=0.004,rounding_size=0.01",
                facecolor=c, edgecolor="white", linewidth=1.2, zorder=3,
            )
        )
        ax.text(
            (x0 + x1) / 2, (ya + yb) / 2, n,
            ha="center", va="center", fontsize=fs,
            color=_ink(c), fontweight="medium", zorder=4,
        )

for col, lab in zip(["L1", "L12", "L2"], ["L1  (5)", "L12  (8)", "L2  (20)"]):
    x0, x1 = cols_x[col]
    ax.text(
        (x0 + x1) / 2, 1.03, lab,
        ha="center", va="bottom", fontsize=11, fontweight="bold", color="#333",
    )

out_png = FIG / "09_ontology_sankey_palette.pdf"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(out_png, bbox_inches="tight", facecolor="white")
plt.show()
print("figures →", out_png)


## 9. 预训练：三级头的内部 CV，以及训练队列上的 StarDist 匹配

这一节的 AUROC 是 **UNI2 A2**（adjacent loss，L2→L1 权重 2，spatial k=8 mean）。`best_mlp` 是 fold 1。ESCC 不在训练表里。

- **内部**：5-fold group CV。`group_cv_folds.csv` 每一折有 L1 head / L12 / L2 macro AUROC，图上是五折均值 ± 标准差。黑点是 `validation_internal_metrics.csv` 的 `oof_l2_macro_auc`，五折验证细胞拼在一起之后的 L2。
- **匹配验证**：同一个 `best_mlp` 在全部 896 张训练切片上的 deployment。StarDist 核中心对上 GT 之后，全体细胞 pooled 的 one-vs-rest（`pooled_roc/roc_stardist_l{1,12,2}_..._per_class_auc.csv`）。Macro = 类间不加权平均。这不是 held-out。
- **分队列**：`metrics_per_section.csv` 的 `macro_auc_*_cells` 是每张切片的 matched-cell macro AUROC。`macro_auc_L1` / `macro_auc_L12` / `macro_auc_L2` 是空的，因为这次预测跳过了 panel 图。图上的刻度是队列内均值。`LungIncomplete` 没有匹配细胞，不进这张图。


In [ ]:
from pathlib import Path
from matplotlib import font_manager as _fm

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

HERE = Path(".").resolve()
if not (HERE / "PanCancerCellType.xlsx").exists():
    HERE = Path("/home/lingyu/ssd2/Python/Hist2Pheno/code/Hist2Pheno_pkg/dataset")
REPO = HERE
for _ in range(8):
    if (REPO / "data" / "PanCancer").is_dir():
        break
    REPO = REPO.parent
else:
    raise FileNotFoundError("data/PanCancer not found above " + str(HERE))

FIG = HERE / "figures_l2_overlap"
FIG.mkdir(exist_ok=True)
PAN = REPO / "data" / "PanCancer"

plt.rcParams["pdf.use14corefonts"] = True
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.use14corefonts": True,
    "font.size": 10,
    "axes.unicode_minus": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

LEVELS = ["L1", "L12", "L2"]
# Three hues from the reference violin (left / middle / right).
LEVEL_COLOR = {"L1": "#7C74A8", "L12": "#3D73B0", "L2": "#4A8F96"}
LEVEL_LABEL = {
    "L1": "L1  coarse",
    "L12": "L12  intermediate",
    "L2": "L2  fine",
}
FOLD_COL = {"L1": "l1_head_macro_auc", "L12": "l12_macro_auc", "L2": "l2_macro_auc"}

UNI2 = "result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42"
UNI2_PRED = "pred_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42_deployment"
folds = pd.read_csv(
    PAN / UNI2 / "cross_dataset_cv/A2_adjacent_spatial_k8mean/group_cv_folds.csv"
)
internal = pd.read_csv(PAN / UNI2 / "validation_internal_metrics.csv").iloc[0]

cv = pd.DataFrame({
    "level": LEVELS,
    "cv_mean_macro_auc": [folds[FOLD_COL[lv]].mean() for lv in LEVELS],
    "cv_std": [folds[FOLD_COL[lv]].std(ddof=0) for lv in LEVELS],
}).set_index("level")
cv.loc["L2", "oof_macro_auc"] = internal["oof_l2_macro_auc"]
print(
    f"UNI2 A2 group CV: {int(internal['n_samples'])} sections, "
    f"{int(internal['n_cells']):,} cells, {int(internal['n_folds'])} folds, "
    f"best_mlp is fold {int(internal['best_fold'])}"
)
display(cv.round(3))

fig, ax = plt.subplots(figsize=(6.4, 3.7))
x = np.arange(len(LEVELS))
vals = cv.loc[LEVELS, "cv_mean_macro_auc"].to_numpy()
errs = cv.loc[LEVELS, "cv_std"].to_numpy()
ax.bar(x, vals, color=[LEVEL_COLOR[lv] for lv in LEVELS], width=0.62,
       yerr=errs, capsize=3, ecolor="#4B5563")
ax.scatter([2], [internal["oof_l2_macro_auc"]], s=36, color="#1B2430", zorder=3,
           label=f"L2 OOF {internal['oof_l2_macro_auc']:.3f}")
for i, v in enumerate(vals):
    ax.text(i, v + 0.025, f"{v:.3f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(["L1 head", "L12", "L2"])
ax.set_ylim(0.55, 1.05)
ax.set_ylabel("macro AUROC")
ax.set_title("UNI2 A2 internal 5-fold group CV")
ax.axhline(0.8, color="#CBD5E1", ls="--", lw=0.8, zorder=0)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "10_internal_three_level_auc.png", bbox_inches="tight", facecolor="white")
plt.show()

per_class = {}
for lv, fname in {
    "L1": "roc_stardist_l1_cross_dataset_all_Complete_Cases_per_class_auc.csv",
    "L12": "roc_stardist_l12_cross_dataset_all_Complete_Cases_per_class_auc.csv",
    "L2": "roc_stardist_l2_cross_dataset_all_Complete_Cases_per_class_auc.csv",
}.items():
    d = pd.read_csv(PAN / UNI2_PRED / "pooled_roc" / fname)
    per_class[lv] = d

ext = pd.DataFrame({
    "pooled_macro_auc": {lv: d["auc_ovr"].mean() for lv, d in per_class.items()},
    "n_classes": {lv: len(d) for lv, d in per_class.items()},
    "n_pos": {lv: int(d["n_pos"].sum()) for lv, d in per_class.items()},
})
print("UNI2 A2 deployment, StarDist matched, pooled across the 896 training sections")
display(ext.round(3))

fig, axes = plt.subplots(1, 3, figsize=(12.8, 5.4), gridspec_kw={"width_ratios": [5, 8, 20]})
for ax, lv in zip(axes, LEVELS):
    d = per_class[lv].sort_values("auc_ovr")
    ax.barh(d["class_name"], d["auc_ovr"], color=LEVEL_COLOR[lv])
    ax.axvline(d["auc_ovr"].mean(), color="#1B2430", ls="--", lw=0.8)
    ax.set_xlim(0.55, 1.01)
    ax.set_title(f"{lv}   macro {d['auc_ovr'].mean():.3f}")
    ax.set_xlabel("one-vs-rest AUROC")
fig.suptitle("UNI2 A2 deployment · StarDist matched · 896 training sections pooled", y=1.02, fontsize=12)
fig.tight_layout()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "10_stardist_pooled_per_class.png", bbox_inches="tight", facecolor="white")
plt.show()



In [ ]:
sec = pd.read_csv(PAN / UNI2_PRED / "metrics_per_section.csv")
sec = sec[sec["n_matched"] > 0].copy()
# UNI2 A2 best_mlp (fold 1) on all 896 matched training sections.
# This is deployment, not the held-out fold. BRCA / CRC / GBM look high
# because most of those slides were in the train folds.
# Figure 2 should use this result dir's fold_plan.csv with fold == 1.
# macro_auc_L* is empty because the deployment run skipped panel figures.
# x-axis n is the number of sections, not cells.
AUC_COL = {"L1": "macro_auc_L1_cells", "L12": "macro_auc_L12_cells", "L2": "macro_auc_L2_cells"}
LEVEL_COLOR = {"L1": "#7C74A8", "L12": "#3D73B0", "L2": "#4A8F96"}

def _fmt_cells(n) -> str:
    n = int(n)
    if n >= 1_000_000:
        s = f"c={n / 1e6:.2f}M"
        return s.replace(".00M", "M")
    if n >= 1000:
        return f"c={n / 1000:.0f}k"
    return f"c={n}"


def _xticklabels(summary: pd.DataFrame) -> list[str]:
    return [
        f"{name}\nn={int(row.n_sections)}\n{_fmt_cells(row.n_matched)}"
        for name, row in summary.iterrows()
    ]


def _annotate_top(ax, x, value, color) -> None:
    """Value sitting above the frame, rotated so the three levels do not collide."""
    ax.text(
        x,
        1.015,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=6.8,
        color=color,
        clip_on=False,
        zorder=5,
    )


def _style_axes(ax, n_groups: int, y0: float = 0.4, y1: float = 1.0) -> None:
    """Outward ticks on both axes; spines stop at the first/last tick."""
    ax.set_xlim(-0.55, n_groups - 0.45)
    ax.set_ylim(y0, 1.22)
    ax.set_yticks(np.arange(y0, y1 + 1e-9, 0.1))
    ax.tick_params(axis="y", direction="out", length=3.6, width=0.75, color="#1F2937", pad=3)
    ax.tick_params(axis="x", direction="out", length=4.0, width=0.75, color="#1F2937", pad=4)
    ax.spines["left"].set_bounds(y0, y1)
    ax.spines["bottom"].set_bounds(0.0, float(n_groups - 1))
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(0.8)
        ax.spines[side].set_color("#1F2937")
    ax.yaxis.grid(False)


def _cohort_stat(stat: str) -> pd.DataFrame:
    """Point estimate only. Keep for tables / median fallback."""
    return (
        sec.groupby("dataset", as_index=True)
        .agg(
            n_sections=("sample", "size"),
            n_matched=("n_matched", "sum"),
            L1=("macro_auc_L1_cells", stat),
            L12=("macro_auc_L12_cells", stat),
            L2=("macro_auc_L2_cells", stat),
        )
    )


def _cohort_summary(how: str = "mean") -> pd.DataFrame:
    """Per-cohort L1/L12/L2. ``how='mean'`` attaches SD; ``how='median'`` attaches IQR."""
    if how not in {"mean", "median"}:
        raise ValueError("how must be 'mean' or 'median'")
    g = sec.groupby("dataset", sort=False)
    out = pd.DataFrame(
        {"n_sections": g["sample"].size(), "n_matched": g["n_matched"].sum()}
    )
    for lv, col in AUC_COL.items():
        if how == "mean":
            out[lv] = g[col].mean()
            out[f"{lv}_err"] = g[col].std(ddof=1)
        else:
            out[lv] = g[col].median()
            out[f"{lv}_err"] = g[col].quantile(0.75) - g[col].quantile(0.25)
        out[f"{lv}_err"] = out[f"{lv}_err"].fillna(0.0)
        out.loc[out["n_sections"] < 2, f"{lv}_err"] = 0.0
    return out.sort_values("L2", ascending=False)


cohort_mean = _cohort_summary("mean")
cohort_median = _cohort_summary("median")  # kept; call _plot_cohort(how="median") if needed
cohort = cohort_mean
print("Per-section macro AUROC. Cohorts ordered by mean L2. Error = SD (mean) or IQR (median).")
display(cohort_mean.round(3))
display(cohort_median.round(3))


def _plot_cohort(how: str = "mean", stem: str | None = None):
    """Grouped bars with error whiskers. Default is mean ± SD. Median uses IQR."""
    summary = _cohort_summary(how)
    if stem is None:
        stem = f"10_stardist_cohort_{how}.png"
    err_name = "SD" if how == "mean" else "IQR"
    fig, ax = plt.subplots(figsize=(4.5, 4.0))
    xx = np.arange(len(summary))
    w = 0.22
    for i, lv in enumerate(LEVELS):
        x = xx + (i - 1) * w
        y = summary[lv].to_numpy()
        e = summary[f"{lv}_err"].to_numpy()
        ax.bar(
            x,
            y,
            w,
            label=LEVEL_LABEL[lv],
            color=LEVEL_COLOR[lv],
            alpha=0.45,
            edgecolor=LEVEL_COLOR[lv],
            linewidth=0.9,
            zorder=2,
        )
        ax.errorbar(
            x,
            y,
            yerr=e,
            fmt="none",
            ecolor="#1F2937",
            elinewidth=0.8,
            capsize=2.2,
            capthick=0.8,
            zorder=3,
        )
        for xi, yi in zip(x, y):
            _annotate_top(ax, xi, yi, LEVEL_COLOR[lv])
    ax.set_xticks(xx)
    ax.set_xticklabels(_xticklabels(summary), fontsize=7.8)
    ax.axhline(0.5, color="#94A3B8", ls=":", lw=0.8, zorder=0)
    ax.axhline(0.8, color="#CBD5E1", ls="--", lw=0.7, zorder=0)
    ax.set_ylabel(f"{'Mean' if how == 'mean' else 'Median'} macro-AUROC (± {err_name})")
    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.30),
        ncol=3,
        fontsize=8,
        handlelength=0.85,
        handletextpad=0.4,
        columnspacing=1.3,
    )
    _style_axes(ax, len(summary))
    fig.tight_layout()
    fig.savefig(FIG / stem, bbox_inches="tight", facecolor="white", dpi=300)
    plt.show()
    return fig


_plot_cohort("mean", "10_stardist_cohort_mean.pdf")
# _plot_cohort("median", "10_stardist_cohort_median.png")
print("figures →", FIG)


In [ ]:
# Same section AUROCs as the bar chart, drawn as violins (reference style).
# Small-n cohorts (n<4) are a strip of points; the rest are filled violins
# with a short mean tick. Median version: _plot_cohort_violin("median").

def _plot_cohort_violin(how: str = "mean", stem: str | None = None):
    if how not in {"mean", "median"}:
        raise ValueError("how must be 'mean' or 'median'")
    if stem is None:
        stem = f"10_stardist_cohort_{how}_violin.png"
    order = list(_cohort_summary("mean").index)
    summary = _cohort_summary("mean")
    fig, ax = plt.subplots(figsize=(4.5, 4.0))
    w = 0.22
    rng = np.random.default_rng(0)
    for i, lv in enumerate(LEVELS):
        col = AUC_COL[lv]
        color = LEVEL_COLOR[lv]
        for j, name in enumerate(order):
            vals = sec.loc[sec["dataset"] == name, col].dropna().to_numpy(dtype=float)
            x0 = j + (i - 1) * w
            n = int(vals.size)
            if n == 0:
                continue
            if n >= 4:
                parts = ax.violinplot(
                    [vals],
                    positions=[x0],
                    widths=w * 0.95,
                    showmeans=False,
                    showmedians=False,
                    showextrema=False,
                )
                body = parts["bodies"][0]
                body.set_facecolor(color)
                body.set_edgecolor(color)
                body.set_alpha(0.38)
                body.set_linewidth(0.9)
            jitter = 0.0 if n >= 12 else 0.035
            xs = np.full(n, x0) + rng.uniform(-jitter, jitter, n)
            ax.scatter(
                xs,
                vals,
                s=9 if n < 8 else 5,
                c=[color],
                alpha=0.55 if n < 8 else 0.22,
                linewidths=0,
                zorder=3,
                rasterized=True,
            )
            loc = float(np.mean(vals) if how == "mean" else np.median(vals))
            ax.hlines(loc, x0 - 0.07, x0 + 0.07, color="#1F2937", lw=1.15, zorder=4)
            _annotate_top(ax, x0, loc, color)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(_xticklabels(summary.loc[order]), fontsize=7.8)
    ax.axhline(0.5, color="#94A3B8", ls=":", lw=0.8, zorder=0)
    ax.axhline(0.8, color="#CBD5E1", ls="--", lw=0.7, zorder=0)
    ax.set_ylabel(f"Section macro-AUROC  ({'mean' if how == 'mean' else 'median'} tick)")
    from matplotlib.patches import Patch
    ax.legend(
        handles=[Patch(facecolor=LEVEL_COLOR[lv], edgecolor=LEVEL_COLOR[lv],
                       alpha=0.45, label=LEVEL_LABEL[lv]) for lv in LEVELS],
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.30),
        ncol=3,
        fontsize=8,
        handlelength=0.85,
        handletextpad=0.4,
        columnspacing=1.3,
    )
    _style_axes(ax, len(order))
    fig.tight_layout()
    fig.savefig(FIG / stem, bbox_inches="tight", facecolor="white", dpi=300)
    plt.show()
    return fig


_plot_cohort_violin("mean")
# _plot_cohort_violin("median")
print("figures →", FIG)


## 10. External validation：Visium NPC 与 BRCA（没有细胞类型 GT）

同一套 UNI2 A2 `best_mlp`，直接读 StarDist 核的 L1 / L12 / L2。这两批切片没有 GT，所以没有 ROC。看组成、空间图，以及由 L2 概率做出的 TLS / SRI / TNI。

| 队列 | 切片 | 输出 |
|---|---|---|
| NPC | `20210809-C-AH4199551`，`20210809-D-AJ3037946` | `data/Visium/NPC/pred_uni2_A2_bestmlp_paired_v2_seed42/` |
| BRCA | `200219`，`200715`，`GSM7782699`，`V1_Breast_Cancer_Block_A_Section_1` | `data/Visium/BRCA/pred_uni2_A2_bestmlp_paired_v2_seed42/` |

流程在 `code/External_val/NPC` 和 `code/External_val/BRCA`。HE 缩放到 0.5 µm/px。UNI2 的 crop 是 14 px（7 µm）。组成从每张切片的 `*_stardist_pred.h5ad` 现算，不读旧的 `composition.csv`。


In [ ]:
try:
    from IPython.display import display, Image, Markdown
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)
    class Image:
        def __init__(self, filename=None, width=None):
            self.filename = filename
    def Markdown(s):
        print(s)
        return s

EXT_PRED = "pred_uni2_A2_bestmlp_paired_v2_seed42"
EXT = {
    "NPC": REPO / "data/Visium/NPC" / EXT_PRED,
    "BRCA": REPO / "data/Visium/BRCA" / EXT_PRED,
}
HEAD_OBS = {
    "L1": ("pred_lineage", "conf_L1"),
    "L12": ("pred_sublineage", "conf_L12"),
    "L2": ("pred_CT", "conf_L2"),
}

def _composition(root: Path) -> pd.DataFrame:
    import anndata as ad

    rows = []
    paths = sorted(root.glob("*/*_stardist_pred.h5ad"))
    if not paths:
        raise FileNotFoundError(f"no UNI2 A2 predictions under {root}")
    for path in paths:
        adata = ad.read_h5ad(path, backed="r")
        try:
            obs = adata.obs
            for head, (lab_col, conf_col) in HEAD_OBS.items():
                frame = pd.DataFrame({
                    "class": obs[lab_col].astype(str).to_numpy(),
                    "conf": obs[conf_col].astype(float).to_numpy(),
                })
                n = max(int(len(frame)), 1)
                for cls, sub in frame.groupby("class", sort=False):
                    cnt = int(len(sub))
                    rows.append({
                        "sample": path.parent.name,
                        "head": head,
                        "class": str(cls),
                        "n": cnt,
                        "fraction": cnt / n,
                        "mean_conf": float(sub["conf"].mean()),
                    })
        finally:
            adata.file.close()
    return pd.DataFrame(rows)

frames = []
for name, root in EXT.items():
    comp = _composition(root)
    comp.insert(0, "cohort", name)
    frames.append(comp)
comp_all = pd.concat(frames, ignore_index=True)

nuclei = (
    comp_all.loc[comp_all["head"] == "L1"]
    .groupby(["cohort", "sample"], as_index=False)["n"]
    .sum()
    .rename(columns={"n": "n_nuclei"})
)
print("StarDist nuclei used for prediction")
display(nuclei)

l1 = (
    comp_all.loc[comp_all["head"] == "L1"]
    .pivot_table(index=["cohort", "sample"], columns="class", values="fraction", aggfunc="sum")
)
l1 = l1.reindex(columns=["Epithelial", "Immune", "Stromal", "Endothelial", "Neural"]).fillna(0)
print("Predicted L1 fraction")
display(l1.round(3))

l2 = comp_all.loc[comp_all["head"] == "L2"].copy()
class_order = l2.groupby("class")["fraction"].mean().sort_values(ascending=False).index.tolist()
sample_ids = (
    comp_all[["cohort", "sample"]]
    .drop_duplicates()
    .sort_values(["cohort", "sample"], ascending=[False, True])
    .reset_index(drop=True)
)
import sys
from matplotlib.lines import Line2D

_repo = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _p in (_repo / "code", _repo / "code" / "Hist2Pheno_pkg"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
from plotting_palettes import get_palette
L2_PAL = get_palette("pancancer", "fine")

present = [name for name in L2_PAL if (l2["class"] == name).any()]
class_order = present + sorted(set(l2["class"]) - set(present))
missing = [name for name in class_order if name not in L2_PAL]
if missing:
    raise RuntimeError("L2 names outside the pan-cancer palette: " + ", ".join(missing))

fig, ax = plt.subplots(figsize=(10.2, 8.2))
y = np.arange(len(class_order))
n_s = len(sample_ids)
h = 0.8 / n_s
sample_labels = []
for i, row in sample_ids.iterrows():
    sub = l2[(l2["cohort"] == row.cohort) & (l2["sample"] == row["sample"])].set_index("class")
    vals = [float(sub.loc[c, "fraction"]) if c in sub.index else 0.0 for c in class_order]
    short = row["sample"].replace("_tissue_image", "")
    sample_labels.append(f"{row.cohort}  {short}")
    ax.barh(
        y + (i - (n_s - 1) / 2) * h, vals, height=h * 0.92,
        color=[L2_PAL[c] for c in class_order],
    )
ax.set_yticks(y)
ax.set_yticklabels(class_order, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("fraction of nuclei")
ax.set_title("UNI2 A2 predicted L2 composition  ·  Visium NPC and BRCA")
ax.text(
    0.0, 1.02, "within each class, top to bottom: " + "  ·  ".join(sample_labels),
    transform=ax.transAxes, ha="left", va="bottom", fontsize=9, color="#333333",
)
handles = [
    Line2D(
        [0], [0], linestyle="none", marker="o", markersize=12,
        markerfacecolor=L2_PAL[name], markeredgecolor="none", label=name,
    )
    for name in class_order
]
ax.legend(
    handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.08),
    fontsize=12, frameon=False, ncol=4,
    handletextpad=0.3, columnspacing=0.9, labelspacing=0.4, borderaxespad=0,
)
fig.tight_layout()
fig.subplots_adjust(bottom=0.28)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig.savefig(FIG / "11_external_l2_composition.png", bbox_inches="tight", facecolor="white")
plt.show()


tls_frames = []
for name, root in EXT.items():
    path = root / "figures" / "tls_score_summary.csv"
    if not path.exists():
        print("missing", path)
        continue
    tab = pd.read_csv(path)
    tab.insert(0, "cohort", name)
    tls_frames.append(tab)
if tls_frames:
    tls = pd.concat(tls_frames, ignore_index=True)
    cols = ["cohort", "sample", "n", "TLS_mean", "TLS_spatial_mean", "pearson", "spearman"]
    print("TLS niche index (abundance vs spatial). SRI / TNI figures are in the same folder.")
    display(tls[cols].round(3))

for name, root in EXT.items():
    figdir = root / "figures"
    show = sorted(figdir.glob("*_spatial_L1_L12_L2.png")) + sorted(figdir.glob("*_TLS_abundance_vs_spatial.png"))
    display(Markdown(f"### {name}"))
    for path in show:
        display(Markdown(f"`{path.relative_to(REPO)}`"))
        display(Image(filename=str(path), width=920))


## 11. 这一版总结覆盖了哪些落盘结果

只核对文件在不在。缺的项会打在表里，方便对一下是不是还没跑完。


In [ ]:
checks = [
    ("注释 celltype_pancancer", HERE / "PanCancerCellType.xlsx"),
    ("UNI2 A2 CV folds", PAN / UNI2 / "cross_dataset_cv/A2_adjacent_spatial_k8mean/group_cv_folds.csv"),
    ("UNI2 A2 OOF", PAN / UNI2 / "validation_internal_metrics.csv"),
    ("UNI2 A2 best_mlp", PAN / UNI2 / "cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt"),
    ("deployment pooled L2 per-class AUC", PAN / UNI2_PRED / "pooled_roc/roc_stardist_l2_cross_dataset_all_Complete_Cases_per_class_auc.csv"),
    ("deployment pooled L1 per-class AUC", PAN / UNI2_PRED / "pooled_roc/roc_stardist_l1_cross_dataset_all_Complete_Cases_per_class_auc.csv"),
    ("deployment per section", PAN / UNI2_PRED / "metrics_per_section.csv"),
    ("NPC TLS summary", REPO / "data/Visium/NPC" / EXT_PRED / "figures/tls_score_summary.csv"),
    ("BRCA TLS summary", REPO / "data/Visium/BRCA" / EXT_PRED / "figures/tls_score_summary.csv"),
]
rows = []
for name, path in checks:
    rel = path.relative_to(REPO) if path.is_relative_to(REPO) else path
    rows.append({"item": name, "on_disk": path.exists(), "path": str(rel)})
status = pd.DataFrame(rows)
display(status)
for cohort, root in EXT.items():
    n_pred = len(list(root.glob("*/*_stardist_pred.h5ad")))
    status.loc[len(status)] = {"item": f"{cohort} UNI2 A2 predictions", "on_disk": n_pred > 0, "path": f"{n_pred} h5ad"}
missing = status.loc[~status["on_disk"], "item"].tolist()
print("missing:", missing if missing else "none")
